# SentinelTex Image Analysis v2: YOLOv8 with real fabric defect photos

**Why v2:** v1 scored mAP50 0.82 on validation (mostly synthetic images) but only 0.09 on the real test photos. The v1 training set had no real tear photos and only 8 real loose thread photos, so the model learned what synthetic defects look like.

**What changes in v2**
1. Adds real fabric defect photos from two public datasets (licences below). Their classes are mapped to ours.
2. Real camo defect photos count 3 times in training.
3. The model is chosen by its score on **real** images only (validation = our real val photos + held out public photos). The synthetic validation images move into training.
4. The confidence cutoff is chosen on validation, never on test.
5. **The test set is exactly the same as v1** (58 real camo photos), so v1 and v2 compare fairly.

**Inputs to add (right panel, Add Input):**

| Input | How to add | Licence |
|---|---|---|
| `sentineltex-dataset` | Your Work (our dataset, same as v1) | ours |
| Fabric Stain Dataset (priemshpathirana) | Search "fabric stain dataset" in Datasets | CC0 1.0 |
| FabriD | Download "Download All" from data.mendeley.com/datasets/vncdwps24r, then upload the zip as a new private Kaggle dataset named `fabrid` | CC BY 4.0 (cite it in the report) |

Then: Accelerator **GPU T4 x2**, Internet **On**, **Save Version → Save & Run All**. About 40 to 70 minutes.

If one public dataset is missing or its format is not recognised, the notebook says so and continues with the rest.

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.is_available())

## 1. Settings

In [ ]:
SEED = 0
EXT_MAX_PER_CLASS = 200     # cap on public images per class, so they do not drown out the camo photos
EXT_MAX_BACKGROUND = 80     # defect free public images
EXT_VAL_FRACTION = 0.15     # share of public images held out for validation
REAL_OVERSAMPLE = 3         # each real camo defect photo appears this many times in training
EPOCHS, PATIENCE = 150, 40
MAX_SIDE = 1024             # public images are shrunk to this size to save time
NAMES = {0: 'stain', 1: 'spot', 2: 'discoloration', 3: 'tear', 4: 'loose_thread', 5: 'pattern_defect'}

## 2. Find our dataset

In [ ]:
import glob, os, shutil, json, random, yaml
random.seed(SEED)
found = [p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), 'images', 'test'))
         and any('_real_' in f for f in os.listdir(os.path.join(os.path.dirname(p), 'images', 'test')))]
assert found, 'sentineltex-dataset not found: add it with Add Input'
OURS = os.path.dirname(found[0])
print('our dataset:', OURS)

## 3. Read the public datasets

Works with the common label formats (COCO json, Pascal VOC xml, YOLO txt with boxes or polygons). Classes are mapped by name; images with a defect we do not use (for example needle marks) are skipped completely so they are not learned as clean fabric.

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

def map_class(name):
    n = str(name).lower().replace('_', ' ').replace('-', ' ')
    if any(k in n for k in ('defect free', 'defectfree', 'no defect', 'good', 'normal')): return 'bg'
    if any(k in n for k in ('hole', 'tear', 'cut')): return 3
    if 'stain' in n: return 0
    if any(k in n for k in ('missing yarn', 'yarn', 'broken', 'thread', 'float')): return 4
    if 'spot' in n: return 1
    return None   # unused class, e.g. needle mark

def poly_to_box(vals):
    xs, ys = vals[0::2], vals[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)

def from_coco(root):
    out = {}
    for jf in glob.glob(f'{root}/**/*.json', recursive=True):
        try: d = json.load(open(jf))
        except Exception: continue
        if not isinstance(d, dict) or not {'images', 'annotations', 'categories'} <= d.keys(): continue
        cats = {c['id']: c['name'] for c in d['categories']}
        imgs = {i['id']: i for i in d['images']}
        folder = os.path.dirname(jf)
        for i in imgs.values():
            p = os.path.join(folder, i['file_name'])
            if not os.path.exists(p):
                hits = glob.glob(f'{root}/**/{os.path.basename(i["file_name"])}', recursive=True)
                if not hits: continue
                p = hits[0]
            out.setdefault(p, [])
        for a in d['annotations']:
            i = imgs.get(a['image_id'])
            if not i or 'bbox' not in a: continue
            p = next((k for k in out if os.path.basename(k) == os.path.basename(i['file_name'])), None)
            if p is None: continue
            x, y, w, h = a['bbox']; W, H = i['width'], i['height']
            out[p].append((map_class(cats[a['category_id']]), (x + w / 2) / W, (y + h / 2) / H, w / W, h / H))
    return out

def from_voc(root):
    out = {}
    for xf in glob.glob(f'{root}/**/*.xml', recursive=True):
        try: r = ET.parse(xf).getroot()
        except Exception: continue
        size = r.find('size'); fn = r.findtext('filename')
        if size is None or not fn: continue
        W, H = float(size.findtext('width') or 0), float(size.findtext('height') or 0)
        stem = os.path.splitext(os.path.basename(xf))[0]
        cand = [os.path.join(os.path.dirname(xf), fn)] + glob.glob(f'{root}/**/{fn}', recursive=True) \
               + [q for e in IMG_EXT for q in glob.glob(f'{root}/**/{stem}{e}', recursive=True)]
        p = next((c for c in cand if os.path.exists(c)), None)
        if p is None: continue
        if not W or not H: W, H = Image.open(p).size
        boxes = []
        for o in r.findall('object'):
            b = o.find('bndbox')
            if b is None: continue
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            boxes.append((map_class(o.findtext('name')), (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H))
        out[p] = boxes
    return out

def yolo_names(start, root):
    d = start
    while d.startswith(root):
        for f in ('data.yaml', 'data.yml'):
            if os.path.exists(os.path.join(d, f)):
                n = (yaml.safe_load(open(os.path.join(d, f))) or {}).get('names')
                if isinstance(n, list): return dict(enumerate(n))
                if isinstance(n, dict): return {int(k): v for k, v in n.items()}
        if os.path.exists(os.path.join(d, 'classes.txt')):
            return dict(enumerate(l.strip() for l in open(os.path.join(d, 'classes.txt')) if l.strip()))
        d = os.path.dirname(d)
    return None

def from_yolo(root):
    out = {}
    for tf in glob.glob(f'{root}/**/*.txt', recursive=True):
        if os.path.basename(tf) == 'classes.txt': continue
        stem = os.path.splitext(os.path.basename(tf))[0]
        img_dir = os.path.dirname(tf).replace('/labels', '/images')
        p = next((os.path.join(d, stem + e) for d in (img_dir, os.path.dirname(tf)) for e in IMG_EXT
                  if os.path.exists(os.path.join(d, stem + e))), None)
        if p is None: continue
        names = yolo_names(os.path.dirname(tf), root)
        if names is None: continue
        boxes, ok = [], True
        for line in open(tf):
            v = line.split()
            if len(v) < 5: continue
            try: c, nums = int(float(v[0])), [float(x) for x in v[1:]]
            except ValueError: ok = False; break
            box = tuple(nums[:4]) if len(nums) == 4 else poly_to_box(nums)
            boxes.append((map_class(names.get(c, c)), *box))
        if ok: out[p] = boxes
    return out

def from_folders(root, already):
    # defect free images that sit in a folder named like 'defect free' / 'good' with no labels
    out = {}
    for e in IMG_EXT:
        for p in glob.glob(f'{root}/**/*{e}', recursive=True):
            if p not in already and map_class(os.path.basename(os.path.dirname(p))) == 'bg':
                out[p] = []
    return out

def read_source(root):
    items = {}
    for reader in (from_coco, from_voc, from_yolo):
        got = reader(root)
        if got:
            print(f'  {reader.__name__}: {len(got)} images')
            items.update(got)
    items.update(from_folders(root, items))
    return items

ext_sources = {}
for root in sorted(glob.glob('/kaggle/input/*')):
    if root.rstrip('/') == OURS.rstrip('/') or OURS.startswith(root + '/'):
        continue
    print('public dataset:', root)
    items = read_source(root)
    if not items:
        print('  WARNING: no labels found in a known format, skipped')
        continue
    ext_sources[os.path.basename(root)] = items

## 4. Pick public images and build the v2 dataset

In [ ]:
from collections import Counter, defaultdict
DATA = '/kaggle/working/dataset_v2'
shutil.rmtree(DATA, ignore_errors=True)
for s in ('train', 'val', 'test'):
    os.makedirs(f'{DATA}/images/{s}'); os.makedirs(f'{DATA}/labels/{s}')

def put(src_img, boxes, split, name, resize=False):
    dst = f'{DATA}/images/{split}/{name}.jpg'
    im = Image.open(src_img).convert('RGB')
    if resize and max(im.size) > MAX_SIDE:
        im.thumbnail((MAX_SIDE, MAX_SIDE))
    im.save(dst, quality=92)
    with open(f'{DATA}/labels/{split}/{name}.txt', 'w') as f:
        for c, x, y, w, h in boxes:
            f.write(f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')

composition = defaultdict(Counter)
# our data: test unchanged; real val stays val; synthetic val moves to train; real defect train photos repeated
for split in ('train', 'val', 'test'):
    for img in sorted(glob.glob(f'{OURS}/images/{split}/*')):
        stem = os.path.splitext(os.path.basename(img))[0]
        lbl = f'{OURS}/labels/{split}/{stem}.txt'
        boxes = [tuple([int(l.split()[0])] + [float(v) for v in l.split()[1:5]]) for l in open(lbl) if l.strip()] if os.path.exists(lbl) else []
        target = 'train' if (split == 'val' and '_synth_' in stem) else split
        copies = REAL_OVERSAMPLE if (target == 'train' and '_real_defect' in stem) else 1
        for k in range(copies):
            put(img, boxes, target, stem if k == 0 else f'{stem}_rep{k}')
        composition[target]['ours_' + ('synthetic' if '_synth_' in stem else 'real')] += 1

# public data: keep images whose boxes all map to our classes, cap per class, split train / val
for src, items in ext_sources.items():
    by_class, bg = defaultdict(list), []
    for p, boxes in items.items():
        if any(b[0] is None for b in boxes): continue                  # has a class we do not use
        boxes = [b for b in boxes if b[0] != 'bg' and b[3] > 0 and b[4] > 0]
        if not boxes:
            bg.append((p, [])); continue
        main = Counter(b[0] for b in boxes).most_common(1)[0][0]
        by_class[main].append((p, boxes))
    chosen = []
    for c, lst in by_class.items():
        random.shuffle(lst); chosen += lst[:EXT_MAX_PER_CLASS]
    random.shuffle(bg); chosen += bg[:EXT_MAX_BACKGROUND]
    random.shuffle(chosen)
    n_val = int(len(chosen) * EXT_VAL_FRACTION)
    for i, (p, boxes) in enumerate(chosen):
        split = 'val' if i < n_val else 'train'
        try:
            put(p, [tuple(b) for b in boxes], split, f'ext_{src[:20]}_{i:04d}', resize=True)
            composition[split]['public_' + src] += 1
        except Exception as e:
            print('skipped', p, e)
    print(src, {NAMES.get(c, c): min(len(v), EXT_MAX_PER_CLASS) for c, v in by_class.items()}, '| defect free:', min(len(bg), EXT_MAX_BACKGROUND))

yaml.safe_dump({'path': DATA, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': NAMES},
               open(f'{DATA}/data.yaml', 'w'), sort_keys=False)
box_counts = {s: Counter() for s in ('train', 'val', 'test')}
for s in box_counts:
    for f in glob.glob(f'{DATA}/labels/{s}/*.txt'):
        for l in open(f):
            if l.strip(): box_counts[s][NAMES[int(l.split()[0])]] += 1
summary = {'images': {k: dict(v) for k, v in composition.items()}, 'boxes': {k: dict(v) for k, v in box_counts.items()}}
print(json.dumps(summary, indent=2))
json.dump(summary, open('/kaggle/working/dataset_v2_composition.json', 'w'), indent=2)

## 5. Train

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data=f'{DATA}/data.yaml', epochs=EPOCHS, imgsz=640, batch=16, patience=PATIENCE,
            seed=SEED, deterministic=True, project='/kaggle/working/runs', name='sentineltex_v2', exist_ok=True,
            fliplr=0.5, flipud=0.5, degrees=10, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, mosaic=1.0,
            plots=True, verbose=False)
BEST = '/kaggle/working/runs/sentineltex_v2/weights/best.pt'
print('saved', BEST)

## 6. Choose the confidence cutoff on validation (real images only)

A photo counts as defective if any box is above the cutoff. We pick the cutoff with the best F1 on validation, then use it unchanged on test.

In [ ]:
best = YOLO(BEST)

def image_level(split, conf):
    tp = fp = tn = fn = 0
    for img in sorted(glob.glob(f'{DATA}/images/{split}/*')):
        lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        truth = os.path.exists(lbl) and open(lbl).read().strip() != ''
        pred = len(best.predict(img, conf=conf, verbose=False)[0].boxes) > 0
        tp += truth and pred; fn += truth and not pred; fp += pred and not truth; tn += (not truth) and (not pred)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {'conf': conf, 'defect_found_correctly': tp, 'defect_missed': fn, 'clean_flagged_wrongly': fp,
            'clean_passed_correctly': tn, 'precision': round(prec, 3), 'recall': round(rec, 3), 'f1': round(f1, 3)}

sweep = [image_level('val', c) for c in (0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5)]
for r in sweep: print(r)
CONF = max(sweep, key=lambda r: (r['f1'], r['conf']))['conf']
print('chosen cutoff (from validation):', CONF)

## 7. Evaluate: validation and the unchanged real test set

The test set has no discoloration or pattern_defect photos, so those two classes are measured on validation only.

In [ ]:
results = {'version': 'v2', 'chosen_conf': CONF, 'val_conf_sweep': sweep}
for split in ['val', 'test']:
    m = best.val(data=f'{DATA}/data.yaml', split=split, imgsz=640, conf=0.001, iou=0.6,
                 project='/kaggle/working/runs', name=f'eval_v2_{split}', exist_ok=True, plots=True, verbose=False)
    per_class = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        per_class[NAMES[int(c)]] = {'precision': round(float(p), 3), 'recall': round(float(r), 3),
                                    'mAP50': round(float(ap50), 3), 'mAP50_95': round(float(ap), 3)}
    results[split] = {'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3),
                      'mAP50': round(float(m.box.map50), 3), 'mAP50_95': round(float(m.box.map), 3),
                      'per_class': per_class}
    print(f'\n=== {split} ===')
    print(f"overall  P {results[split]['precision']}  R {results[split]['recall']}  mAP50 {results[split]['mAP50']}  mAP50-95 {results[split]['mAP50_95']}")
    for k, v in per_class.items():
        print(f"{k:15s} P {v['precision']}  R {v['recall']}  mAP50 {v['mAP50']}")

results['test_image_level_at_chosen_conf'] = image_level('test', CONF)
results['test_image_level_at_0.25'] = image_level('test', 0.25)     # same cutoff as v1, for comparison
print('\ntest, chosen cutoff:', results['test_image_level_at_chosen_conf'])
print('test, cutoff 0.25 (as v1):', results['test_image_level_at_0.25'])
json.dump(results, open('/kaggle/working/metrics_v2.json', 'w'), indent=2)

## 8. Package the results

In [ ]:
os.makedirs('/kaggle/working/examples_v2', exist_ok=True)
for img in sorted(glob.glob(f'{DATA}/images/test/*'))[:12]:
    best.predict(img, conf=CONF, save=True, project='/kaggle/working/examples_v2', name='pred', exist_ok=True, verbose=False)
out = '/kaggle/working/package_v2'
os.makedirs(out, exist_ok=True)
shutil.copy(BEST, f'{out}/best.pt')
for f in ('metrics_v2.json', 'dataset_v2_composition.json'):
    shutil.copy(f'/kaggle/working/{f}', out)
for f in ['results.csv', 'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png', 'labels.jpg']:
    p = f'/kaggle/working/runs/sentineltex_v2/{f}'
    if os.path.exists(p): shutil.copy(p, out)
for split in ['val', 'test']:
    p = f'/kaggle/working/runs/eval_v2_{split}/confusion_matrix.png'
    if os.path.exists(p): shutil.copy(p, f'{out}/confusion_matrix_{split}.png')
shutil.copytree('/kaggle/working/examples_v2/pred', f'{out}/examples', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/sentineltex_yolo_results_v2', 'zip', out)
print('Download sentineltex_yolo_results_v2.zip from the Output panel')